In [3]:
import polars as pl
import polars.selectors as cs
import polars_ols as pls

%load_ext autoreload
%autoreload 2

In [4]:
sigs = pl.read_parquet('data/eu_signatures_1q.parquet')
sigs.select(pl.mean_horizontal(cs.numeric().abs().lt(0.001).sum() / pl.len()))
sigs.select(pl.col('^.*amount.*$'))#.collect()

"S(JPY,open.amount)","S(GBP,low.amount)","S(HUF,low.amount)","S(CHF,open.amount)","S(AUD,open.amount)","S(NZD,open.amount)","S(open.amount,Spain_News_Index)","S(high.amount,Spain_News_Index)","S(low.amount,Spain_News_Index)","S(low.amount,USD)","S(low.amount,JPY)","S(low.amount,NZD)","S(low.amount,low.amount)","S(low.amount,volume)","S(close.amount,Spain_News_Index)","S(close_unadj.amount,Spain_News_Index)","S(volume,low.amount)"
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.000044,-0.000076,0.000014,0.000041,0.000018,0.000003,0.0,0.0,0.0,-0.000046,-0.000064,-0.000005,0.000539,0.001059,0.0,0.0,0.001059
0.000003,-0.000096,0.000027,-0.000005,0.000005,0.000035,0.0,0.0,0.0,-0.000074,-0.000018,0.000216,0.000981,-0.021078,0.0,0.0,-0.001486
-0.000032,-0.000134,-0.000016,-0.000037,-0.000013,0.000064,0.0,0.0,0.0,-0.000282,-0.000274,0.000171,0.000622,-0.014112,0.0,0.0,0.002316
-0.000109,0.000048,0.000199,-0.000123,-0.000071,0.000144,0.0,0.0,0.0,-0.00045,0.000049,0.000309,0.001599,-0.023461,0.0,0.0,-0.006967
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.000741,-0.000812,-0.001776,-0.00042,-0.000096,-0.000171,0.0,0.0,0.0,-0.001963,0.001252,-0.001936,0.003201,0.481928,0.0,0.0,0.349248
-0.000344,-0.000354,-0.002183,-0.000987,-0.001353,-0.000973,0.0,0.0,0.0,-0.002159,0.001187,-0.001904,0.00111,0.52409,0.0,0.0,-0.024382
-0.000539,-0.00029,-0.002232,0.000378,0.001899,0.00028,0.0,0.0,0.0,-0.002437,0.001044,-0.001666,0.000931,0.514395,0.0,0.0,-0.066071


In [6]:
from extract.io import get_stocks
import dvc.api
sigs = pl.scan_parquet('data/eu_signatures_1q.parquet')
stocks = get_stocks(dvc.api.params_show()['stocks']['defence'])
df = sigs.join(stocks.lazy(), on=['ticker', 'date']).drop_nulls().sort(['ticker', 'date'])
df, stats = pl.collect_all([df, sigs.group_by('ticker').agg(years_of_data=pl.len() // 365)])
print(stats)
df


shape: (13, 2)
┌───────────┬───────────────┐
│ ticker    ┆ years_of_data │
│ ---       ┆ ---           │
│ str       ┆ u32           │
╞═══════════╪═══════════════╡
│ HAG.DE    ┆ 3             │
│ AIR.PA    ┆ 7             │
│ AVIO.MI   ┆ 6             │
│ HO.PA     ┆ 7             │
│ LDO.MI    ┆ 6             │
│ …         ┆ …             │
│ SAF.PA    ┆ 7             │
│ SAAB-B.ST ┆ 6             │
│ BA.L      ┆ 6             │
│ R3NK.DE   ┆ 1             │
│ RHM.DE    ┆ 6             │
└───────────┴───────────────┘


ticker,date,S(volume),"S(JPY,open.amount)","S(GBP,low.amount)","S(HUF,low.amount)","S(CHF,open.amount)","S(AUD,open.amount)","S(NZD,open.amount)","S(open.amount,Spain_News_Index)","S(high.amount,Spain_News_Index)","S(low.amount,Spain_News_Index)","S(low.amount,USD)","S(low.amount,JPY)","S(low.amount,NZD)","S(low.amount,low.amount)","S(low.amount,volume)","S(close.amount,Spain_News_Index)","S(close_unadj.amount,Spain_News_Index)","S(volume,low.amount)",close.amount
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,"decimal[38,10]"
"""AIR.PA""",2016-04-11,0.634169,-0.001112,-0.000803,-0.000809,0.000069,0.000031,0.000467,0.0,0.0,0.0,0.000214,0.001414,-0.00061,0.000245,0.005336,0.0,0.0,-0.019377,48.9100000000
"""AIR.PA""",2016-04-12,0.78143,-0.000726,-0.00128,-0.000927,0.000196,-0.000133,0.000412,0.0,0.0,0.0,0.000197,0.001372,-0.000355,0.000955,0.000489,0.0,0.0,-0.034635,48.6700000000
"""AIR.PA""",2016-04-13,0.952915,-0.00095,-0.000701,-0.00082,0.000124,-0.000132,0.000344,0.0,0.0,0.0,0.000434,0.001365,-0.000017,0.000065,-0.004234,0.0,0.0,-0.006614,50.0000000000
"""AIR.PA""",2016-04-14,0.920236,-0.001224,-0.000845,-0.000848,0.000044,-0.000238,0.000216,0.0,0.0,0.0,0.000498,0.001414,-0.000073,0.000207,-0.003716,0.0,0.0,-0.01499,49.3600000000
"""AIR.PA""",2016-04-15,0.698876,-0.000708,-0.000831,-0.000846,0.000153,0.000024,0.000438,0.0,0.0,0.0,0.000442,0.001443,6.8475e-7,0.00019,0.000693,0.0,0.0,-0.014326,49.1100000000
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SAF.PA""",2026-03-11,10.388878,-0.000741,-0.000812,-0.001776,-0.00042,-0.000096,-0.000171,0.0,0.0,0.0,-0.001963,0.001252,-0.001936,0.003201,0.481928,0.0,0.0,0.349248,318.2000000000
"""SAF.PA""",2026-03-12,10.606215,-0.000344,-0.000354,-0.002183,-0.000987,-0.001353,-0.000973,0.0,0.0,0.0,-0.002159,0.001187,-0.001904,0.00111,0.52409,0.0,0.0,-0.024382,308.1000000000
"""SAF.PA""",2026-03-13,10.391368,-0.000539,-0.00029,-0.002232,0.000378,0.001899,0.00028,0.0,0.0,0.0,-0.002437,0.001044,-0.001666,0.000931,0.514395,0.0,0.0,-0.066071,304.4000000000


In [9]:
stocks.schema

Schema([('ticker', String),
        ('date', Date),
        ('close.amount', Decimal(precision=38, scale=10))])

In [10]:
from statsforecast.models import AutoARIMA
from statsforecast import StatsForecast

date = pl.col.date.cast(pl.Datetime)
sf = StatsForecast(models=[AutoARIMA()], freq='1QS')#date.dt.offset_by('QS'))
stock_df = stocks.select(date, 'ticker', cs.numeric()).to_pandas()
stock_df[['open.amount', 'high.amount', 'low.amount', 'close.amount', 'close_unadj.amount']] = stock_df[['open.amount', 'high.amount', 'low.amount', 'close.amount', 'close_unadj.amount']].astype(float)
sf.forecast(df=stock_df[['date', 'ticker', 'close.amount']], X_df=stock_df, h=30, time_col='date', target_col='close.amount', id_col='ticker', fitted=True)
#forecast.fit(stocks.select(date, 'ticker', cs.numeric()), time_col='date', target_col='close.amount', id_col='ticker')

ModuleNotFoundError: No module named 'statsforecast'

S(open.amount),"S(close_unadj.amount,open.amount)","S(low.amount,open.amount)","S(European_News_Index,open.amount)","S(UK_News_Index,open.amount)","S(Spain_News_Index,open.amount)","S(Germany_News_Index,open.amount)","S(open.amount,close_unadj.amount)","S(open.amount,low.amount)","S(open.amount,European_News_Index)","S(open.amount,UK_News_Index)","S(open.amount,Spain_News_Index)","S(open.amount,Germany_News_Index)","S(open.amount,open.amount)","S(open.amount,Italy_News_Index)","S(open.amount,ZAR)","S(open.amount,France_News_Index)","S(open.amount,high.amount)","S(open.amount,close.amount)","S(open.amount,volume)","S(Italy_News_Index,open.amount)","S(France_News_Index,open.amount)","S(high.amount,open.amount)","S(close.amount,open.amount)","S(volume,open.amount)",open.amount
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,"decimal[38,10]"
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,52.4900000000
-0.007811,0.000008,0.000022,0.0,0.0,0.0,0.0,0.000008,0.000022,0.0,0.0,0.0,0.0,0.000031,0.0,-0.000111,0.0,0.000024,0.000007,-0.00007,0.0,0.0,0.000024,0.000007,-0.00007,52.0800000000
-0.001091,0.000022,0.000008,0.0,0.0,0.0,0.0,-0.000028,-0.000009,0.0,0.0,0.0,0.0,5.9468e-7,0.0,-0.000193,0.0,-0.000004,-0.000028,0.000177,0.0,0.0,0.000004,0.000022,-0.000136,52.4300000000
0.007683,-0.000015,-0.00017,0.0,0.0,0.0,0.0,-0.000096,-0.000152,0.0,0.0,0.0,0.0,0.00003,0.0,-0.00029,0.0,0.000007,-0.000096,0.005007,0.0,0.0,0.000018,-0.000015,0.005964,52.8900000000
-0.015006,0.000109,0.000408,0.0,0.0,0.0,0.0,-0.000162,-0.000272,0.0,0.0,0.0,0.0,0.000113,0.0,-0.000245,0.0,0.000028,-0.000162,0.004771,0.0,0.0,0.000008,0.000107,-0.027166,51.6900000000
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.107405,0.013432,0.008949,0.0,0.0,0.0,0.0,-0.001883,0.001728,0.0,0.0,0.0,0.0,0.005768,0.0,-0.001179,0.0,0.001408,-0.001883,0.514252,0.0,0.0,0.009899,0.013432,0.640539,323.8000000000
0.089065,0.010979,0.006273,0.0,0.0,0.0,0.0,-0.002178,0.000853,0.0,0.0,0.0,0.0,0.003966,0.0,-0.001236,0.0,0.000124,-0.002178,0.549241,0.0,0.0,0.007908,0.010979,0.376044,316.1000000000
0.098505,0.011242,0.006981,0.0,0.0,0.0,0.0,-0.00521,-0.00234,0.0,0.0,0.0,0.0,0.004852,0.0,-0.000689,0.0,0.000501,-0.00521,0.593636,0.0,0.0,0.009114,0.011242,0.451125,319.3000000000


In [5]:
import infomeasure as im
from tqdm import tqdm
valid_cols = list()
horizon = 30
for col in tqdm(cs.expand_selector(df, cs.matches('^S(.*)$'))):
    res = im.mutual_information(df.get_column(col).slice(horizon), df.get_column('close.amount').diff(horizon).slice(horizon), approach='ksg')
    if res > 0.1:
        valid_cols.append(col)
len(valid_cols)

100%|██████████| 252/252 [01:13<00:00,  3.42it/s]


12

In [6]:
valid_cols

['S(close_unadj.amount)',
 'S(low.amount)',
 'S(open.amount)',
 'S(high.amount)',
 'S(close.amount)',
 'S(volume)',
 'S(volume,close_unadj.amount)',
 'S(volume,low.amount)',
 'S(volume,open.amount)',
 'S(volume,high.amount)',
 'S(volume,close.amount)',
 'S(volume,volume)']

In [137]:
print(df.shape)
df = df.select('date', 'ticker', *valid_cols, 'close.amount')
print(df.shape)

(29902, 265)
(29902, 15)


In [141]:
pl.col.y.diff(30).least_squares.lasso

<bound method LeastSquares.lasso of <polars_ols.LeastSquares object at 0x3648c84c0>>

In [12]:
df.select(pl.col('close.amount').diff(30).least_squares.lasso(*valid_cols, mode='coefficients', alpha=1.0, null_policy='drop').over('ticker'))

ComputeError: the plugin panicked

The message is suppressed. Set POLARS_VERBOSE=1 to send the panic message to stderr.

In [138]:
from sklearn.linear_model import Lasso
duration = pl.col.date.max() - pl.col.date.min()
n_blocks = 10
max_lag = 1
#X = cs.numeric()#.exclude('close.amount')
X = cs.numeric()
#y = [pl.col('close.amount').diff().shift(-i).name.suffix(f"<<{i}") for i in range(1, max_lag, 1)]
y = pl.col('close.amount').diff().shift(-max_lag).fill_null(0.0)
for idx in range(n_blocks // 2, n_blocks, n_blocks // 4):
    anchor = pl.col.date.min() + (duration / n_blocks * idx)
    train, test = df.filter(pl.col.date.lt(anchor)), df.filter(pl.col.date.gt(anchor))
    print(pl.concat([train.select(min=pl.col.date.min(), max=pl.col.date.max()), test.select(min=pl.col.date.min(), max=pl.col.date.max())]))

    regr = Lasso(alpha=1)
    X_train, X_test = train.select(X).slice(max_lag), test.select(X).slice(max_lag)
    y_train, y_test = train.select(y).head(-max_lag), test.select(y).head(-max_lag) 
    regr.fit(X_train, y_train)
    print(regr.score(X_test, y_test))

shape: (2, 2)
┌────────────┬────────────┐
│ min        ┆ max        │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2016-03-14 ┆ 2021-03-12 │
│ 2021-03-15 ┆ 2026-03-13 │
└────────────┴────────────┘


/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


0.0007347415054995698
shape: (2, 2)
┌────────────┬────────────┐
│ min        ┆ max        │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2016-03-14 ┆ 2023-03-10 │
│ 2023-03-14 ┆ 2026-03-13 │
└────────────┴────────────┘


/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


0.0009384363668981655
shape: (2, 2)
┌────────────┬────────────┐
│ min        ┆ max        │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2016-03-14 ┆ 2025-03-11 │
│ 2025-03-13 ┆ 2026-03-13 │
└────────────┴────────────┘


/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/remiadon/strats/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


0.0015329769484596634


In [56]:
import polars.selectors as cs
part = pl.col.date.dt.to_string().gt('2023-01-01')
max_lag = 3
X = [pl.col('close.amount').pct_change().shift(i).name.suffix(f'>>{i}').cast(pl.Float64) for i in range(max_lag * 4)]
y = [pl.col('close.amount').pct_change().shift(-i).name.suffix(f'<<{i}').cast(pl.Float64) for i in range(max_lag)]
tmp = stocks.with_columns(part=part).with_columns(*X, *y).drop_nulls()
a = max_lag * 4 + 1
b = max_lag + 1
print(train.select(min=pl.col.date.min(), max=pl.col.date.max()))
print(test.select(min=pl.col.date.min(), max=pl.col.date.max()))
train.shape, test.shape

shape: (1, 2)
┌────────────┬────────────┐
│ min        ┆ max        │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2016-03-14 ┆ 2025-03-11 │
└────────────┴────────────┘
shape: (1, 2)
┌────────────┬────────────┐
│ min        ┆ max        │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2025-03-13 ┆ 2026-03-13 │
└────────────┴────────────┘


((26575, 31), (3314, 31))

In [160]:
from itertools import compress
tmp = df.select(cs.numeric().eq(0.0).sum().truediv(pl.len()).gt(0.9))#.collect(engine='streaming')
len(list(compress(*tmp.transpose(include_header=True))))

1755

In [208]:
signature(pl.col.UK_News_Index_policy_uncertainty)

{('UK_News_Index_policy_uncertainty',): <Expr ['col("UK_News_Index_policy_unce…'] at 0x4688D7B80>,
 ('UK_News_Index_policy_uncertainty',
  'UK_News_Index_policy_uncertainty'): <Expr ['[([([(dyn float: 0.5) * (col("…'] at 0x4688D4130>}

In [8]:
pl.read_parquet('data/eu_uncertainty.parquet', columns=['date', 'UK_News_Index_policy_uncertainty'])\
    .filter(pl.col.date.dt.to_string().gt('2020-01-01') & pl.col.date.dt.to_string().lt('2020-09-01'))\
    .upsample(time_column='date', every='1d')\
    .rolling(period='45d', index_column='date').agg(
        **{f"S({','.join(k)})": expr for k, expr in signature(pl.col.UK_News_Index_policy_uncertainty).items()}
        #pl.col.UK_News_Index_policy_uncertainty.interpolate_by('date').forward_fill().diff().fill_null(0.0).cum_sum()
    )#.select(pl.exclude('date').implode())#.to_dicts()

date,S(UK_News_Index_policy_uncertainty),"S(UK_News_Index_policy_uncertainty,UK_News_Index_policy_uncertainty)"
date,list[f64],list[f64]
2020-02-01,[0.0],[0.0]
2020-02-02,[0.0],[0.0]
2020-02-03,[0.0],[0.0]
2020-02-04,[0.0],[0.0]
2020-02-05,[0.0],[0.0]
…,…,…
2020-07-28,[0.0],[0.0]
2020-07-29,[0.0],[0.0]
2020-07-30,[0.0],[0.0]


In [164]:
df.select('date', pl.col('S(UK_News_Index_policy_uncertainty)_1q').eq(0).sum() / pl.len())

date,S(UK_News_Index_policy_uncertainty)_1q
date,f64
2021-09-07,1.0
2021-09-14,1.0
2021-09-23,1.0
2021-10-05,1.0
2021-10-18,1.0
…,…
2021-06-28,1.0
2021-07-15,1.0
2021-07-29,1.0


In [37]:
import polars_ols as pls
import polars.selectors as cs
res = df.group_by('ticker').agg(pl.col('close.amount').shift(-1).least_squares.lasso(cs.numeric(), alpha=1, mode='statistics'))
res

ComputeError: the plugin panicked

The message is suppressed. Set POLARS_VERBOSE=1 to send the panic message to stderr.

In [27]:
df.group_by('ticker').agg(pl.col.date.min())

ticker,date
str,date
"""HO.PA""",2016-03-14


In [36]:
df.null_count().transpose().filter(pl.col.column_0.ne(0))

column_0
u32
